# Notebook 1 — Data Understanding & Cleaning
## AI Job Market Intelligence & Smart Recommender

**Objective:** Load the raw Naukri job-postings scrape, understand its structure and quality issues, and produce a cleaned, parsed dataset (`naukri_cleaned.csv`) that downstream notebooks can rely on without re-deriving cleaning logic.

**Scope of this notebook:**
1. Load raw data and standardize column names
2. Assess data quality: duplicates, missing values, types
3. Parse `job_salary` into structured numeric fields
4. Parse `job_experience_required` into structured numeric fields
5. Clean `location` and `key_skills` text fields
6. Run sanity-check assertions on the cleaned output
7. Save the cleaned dataset

**Explicitly out of scope for this notebook** (deferred to Notebook 3 — Feature Engineering, and documented there): statistical outlier removal, city/industry bucketing for modeling, and any transformation that should be fit only on a training split. Doing that here, before the train/test split exists, would risk data leakage — so this notebook only *parses and structures* the data; it does not make modeling decisions.

**Reference:** Column definitions are maintained centrally in `docs/02_Data_Dictionary.md` — refer there rather than re-deriving meaning from code.

**Source data:** Naukri.com job postings, 2019 scrape, ~30,000 rows (public Kaggle dataset). See `docs/01_Project_Charter_BRD.md` for full source disclosure and the "2019 data, not current market rates" limitation.


## 1. Import Libraries

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)


## 2. Load Raw Dataset

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "dataset" / "naukri_dataset.csv"

df = pd.read_csv(DATA_PATH)
print("Shape of raw dataset:", df.shape)
df.head()


Shape of raw dataset: (30000, 11)


,Uniq Id,Crawl Timestamp,Job Title,Job Salary,Job Experience Required,Key Skills,Role Category,Location,Functional Area,Industry,Role
0,9be62c49a0b7ebe982a4af1edaa7bc5f,2019-07-05 01:46:07 +0000,Digital Media Planner,Not Disclosed by Recruiter,5 - 10 yrs,Media Planning| Digital Media,Advertising,Mumbai,"Marketing , Advertising , MR , PR , Media Planning","Advertising, PR, MR, Event Management",Media Planning Executive/Manager
1,3c52d436e39f596b22519da2612f6a56,2019-07-06 08:04:50 +0000,Online Bidding Executive,Not Disclosed by Recruiter,2 - 5 yrs,pre sales| closing| software knowledge| clients| requirements| negotiating| client| online bidding| good communication| technology,Retail Sales,"Pune,Pune","Sales , Retail , Business Development","IT-Software, Software Services",Sales Executive/Officer
2,ffad8a2396c60be2bf6d0e2ff47c58d4,2019-08-05 15:50:44 +0000,Trainee Research/ Research Executive- Hi- Tech Operations,Not Disclosed by Recruiter,0 - 1 yrs,Computer science| Fabrication| Quality check| Intellectual property| Electronics| Support services| Research| Management| Human resource management| Research Executive,R&D,Gurgaon,"Engineering Design , R&D","Recruitment, Staffing",R&D Executive
3,7b921f51b5c2fb862b4a5f7a54c37f75,2019-08-05 15:31:56 +0000,Technical Support,"2,00,000 - 4,00,000 PA.",0 - 5 yrs,Technical Support,Admin/Maintenance/Security/Datawarehousing,Mumbai,"IT Software - Application Programming , Maintenance","IT-Software, Software Services",Technical Support Engineer
4,2d8b7d44e138a54d5dc841163138de50,2019-07-05 02:48:29 +0000,Software Test Engineer -hyderabad,Not Disclosed by Recruiter,2 - 5 yrs,manual testing| test engineering| test cases| web testing| web technologies,Programming & Design,Hyderabad,IT Software - QA & Testing,"IT-Software, Software Services",Testing Engineer


## 3. Standardize Column Names

Raw column names are inconsistent (mixed case, spaces). Standardizing to `snake_case` up front avoids repeated `.str.strip().lower()` calls scattered through later notebooks.

In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Standardized columns:\n", df.columns.tolist())


Standardized columns:
 ['uniq_id', 'crawl_timestamp', 'job_title', 'job_salary', 'job_experience_required', 'key_skills', 'role_category', 'location', 'functional_area', 'industry', 'role']


## 4. Initial Data Quality Assessment

Before touching any values, establish a baseline: duplicates, dtypes, and missingness. This baseline is referenced again at the end of the notebook to confirm cleaning didn't silently drop more data than expected.

In [4]:
print("Duplicate rows (exact duplicates across all columns):", df.duplicated().sum())
print("Duplicate uniq_id values:", df["uniq_id"].duplicated().sum())


Duplicate rows (exact duplicates across all columns): 0
Duplicate uniq_id values: 0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   uniq_id                  30000 non-null  object
 1   crawl_timestamp          30000 non-null  object
 2   job_title                29425 non-null  object
 3   job_salary               29950 non-null  object
 4   job_experience_required  29427 non-null  object
 5   key_skills               28729 non-null  object
 6   role_category            27695 non-null  object
 7   location                 29423 non-null  object
 8   functional_area          29427 non-null  object
 9   industry                 29427 non-null  object
 10  role                     29099 non-null  object
dtypes: object(11)
memory usage: 2.5+ MB


In [6]:
# Parse crawl timestamp to proper datetime
df["crawl_timestamp"] = pd.to_datetime(df["crawl_timestamp"], errors="coerce")

print("Unparseable timestamps:", df["crawl_timestamp"].isna().sum())
print("Date range:", df["crawl_timestamp"].min(), "to", df["crawl_timestamp"].max())


Unparseable timestamps: 0
Date range: 2019-07-04 01:26:32+00:00 to 2019-08-06 09:24:54+00:00


In [7]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": (missing / len(df) * 100).round(2)
})

missing_df


,Missing Count,Missing %
role_category,2305,7.68
key_skills,1271,4.24
role,901,3.00
location,577,1.92
job_title,575,1.92
functional_area,573,1.91
job_experience_required,573,1.91
industry,573,1.91
job_salary,50,0.17
crawl_timestamp,0,0.00


**Observation:** `job_salary` will show 0% technically-missing here because "Not Disclosed by Recruiter" is a string value, not a null — the real disclosure rate is only visible after parsing (Section 5). This is called out explicitly so the missingness table above isn't misread as "salary is mostly present."


## 5. Salary Cleaning

**Goal:** Convert the free-text `job_salary` field into structured numeric fields, and flag which postings actually disclose a salary at all.

**Known formats in the raw data:** "Not Disclosed by Recruiter", single values ("3,00,000 PA"), and ranges ("2,00,000 - 3,50,000 PA"), with amounts sometimes expressed in Lakhs.

**Documented limitation:** the source data does not reliably distinguish monthly vs. annual salary in all rows — we assume annual (PA / per-annum) since that's the dominant convention on Naukri postings, but this is an assumption, not a verified fact, and is stated here rather than silently baked into the numbers.

In [8]:
df["job_salary"] = df["job_salary"].str.strip()

# Flag: was an actual salary figure disclosed, or is this a placeholder string?
df["salary_disclosed"] = np.where(
    df["job_salary"].str.contains("not disclosed", case=False, na=False),
    0,
    1
)

print(df["salary_disclosed"].value_counts(normalize=True).round(3))


salary_disclosed
0    0.692
1    0.308
Name: proportion, dtype: float64


In [9]:
def extract_salary(salary):
    """
    Parse a free-text salary string into (min, max) numeric annual INR values.
    Returns (NaN, NaN) if undisclosed or unparseable.

    Known limitation: assumes all disclosed figures are annual (PA), consistent
    with Naukri's dominant posting convention. Not independently verified per row.
    """
    if pd.isna(salary):
        return np.nan, np.nan

    salary = salary.lower().strip()

    if "not disclosed" in salary:
        return np.nan, np.nan

    salary = salary.replace(",", "")
    numbers = re.findall(r'\d+', salary)

    if len(numbers) == 0:
        return np.nan, np.nan

    is_lakh = "lakh" in salary or "lac" in salary

    # Implausibility ceiling: catches garbage the regex would otherwise accept as a
    # "salary" — phone numbers embedded in free text (e.g. "Call - 9840040143"),
    # and unrelated figures in contaminated entries (pay-scale tables, incentive
    # descriptions). No legitimate annual salary in this dataset should exceed
    # roughly Rs. 1 crore; anything above that is treated as a parsing failure,
    # not a real high earner. This is a plausibility check, distinct from the
    # statistical outlier capping done later in Notebook 3 — this one exists to
    # catch data that was never a salary in the first place.
    PLAUSIBLE_MAX_ANNUAL_SALARY = 10_000_000  # Rs. 1 crore

    if len(numbers) == 1:
        value = int(numbers[0])
        if is_lakh:
            value *= 100000
        if value < 10000 or value > PLAUSIBLE_MAX_ANNUAL_SALARY:
            return np.nan, np.nan
        return value, value

    min_sal = int(numbers[0])
    max_sal = int(numbers[1])

    if is_lakh:
        min_sal *= 100000
        max_sal *= 100000

    if min_sal < 10000 or max_sal < 10000:
        return np.nan, np.nan

    if min_sal > PLAUSIBLE_MAX_ANNUAL_SALARY or max_sal > PLAUSIBLE_MAX_ANNUAL_SALARY:
        return np.nan, np.nan

    # Defensive check: some raw entries have stray trailing digits after the
    # real figure (e.g. "INR 9,00,000 PA. 885804" — an unrelated number
    # concatenated onto the salary string). If that produces min > max, we
    # cannot safely infer which number was the intended one, so we treat the
    # whole entry as unparseable rather than silently guessing.
    if min_sal > max_sal:
        return np.nan, np.nan

    return min_sal, max_sal


df[["salary_min", "salary_max"]] = df["job_salary"].apply(
    lambda x: pd.Series(extract_salary(x))
)

df["salary_avg"] = (df["salary_min"] + df["salary_max"]) / 2

parsed_rate = df["salary_min"].notna().mean()
print(f"Successfully parsed a numeric salary for {parsed_rate:.1%} of all postings")
print(f"(vs. {df['salary_disclosed'].mean():.1%} flagged as 'disclosed' by the raw text)")


Successfully parsed a numeric salary for 26.8% of all postings
(vs. 30.8% flagged as 'disclosed' by the raw text)


**Data quality note:** a small number of raw entries contain corrupted or non-salary text — e.g. a stray number concatenated after the real salary figure (`"INR 9,00,000 PA. 885804"`), or free text with no real salary at all (phone numbers left in the field, e.g. `"Call - 9840040143"`, or pay-scale/incentive descriptions). Two defenses are applied:
1. If parsing produces `min > max`, the entry is marked unparseable (can't safely guess intent).
2. If the parsed value exceeds a plausible annual salary ceiling (Rs. 1 crore), it's rejected as garbage rather than accepted as a legitimate high earner — this specifically catches phone numbers and unrelated figures in contaminated free text, which would otherwise parse as salaries in the tens or hundreds of millions.

Both were caught by the sanity-check assertions in Section 10 — the first during the initial build of this notebook, the second only surfaced once Notebook 2's statistical tests came back with a correlation near zero, which was the actual signal that something was still wrong. That's worth stating plainly: passing internal-consistency checks (min ≤ max, non-negative) is necessary but not sufficient — a magnitude-plausibility check was needed too, and is now part of Section 10.

**Insight:** the parsed rate should be close to, but not exactly equal to, the `salary_disclosed` flag rate — any gap represents postings that claimed to disclose a salary but were in a format `extract_salary` couldn't parse. That gap is quantified above rather than silently absorbed as missing data.

**Note on outliers:** we deliberately do **not** cap or remove salary outliers in this notebook. Outlier handling is a modeling decision and is performed in Notebook 3, *after* the train/test split, so that no information about the distribution's extremes leaks from test data into decisions made about the training data.

## 6. Experience Cleaning

**Goal:** Convert `job_experience_required` free text (e.g. "2 - 5 yrs") into structured min/max/avg numeric years.

In [10]:
def extract_experience(exp):
    """
    Parse a free-text experience string into (min, max) years.
    Handles both range format ("2 - 5 yrs") and single-value format ("5 yrs").
    Caps implausible values (>40 years) as unparseable rather than silently keeping them.
    """
    if pd.isna(exp):
        return np.nan, np.nan

    exp = exp.lower().strip()
    numbers = re.findall(r'\d+', exp)

    if len(numbers) == 0:
        return np.nan, np.nan

    if len(numbers) == 1:
        val = int(numbers[0])
        if val > 40:
            return np.nan, np.nan
        return val, val

    min_exp = int(numbers[0])
    max_exp = int(numbers[1])

    if min_exp > 40 or max_exp > 40 or min_exp > max_exp:
        return np.nan, np.nan

    return min_exp, max_exp


df[["exp_min", "exp_max"]] = df["job_experience_required"].apply(
    lambda x: pd.Series(extract_experience(x))
)

df["exp_avg"] = (df["exp_min"] + df["exp_max"]) / 2

print("Unparseable experience values:", df["exp_avg"].isna().sum(),
      f"({df['exp_avg'].isna().mean():.1%})")
df["exp_avg"].describe()


Unparseable experience values: 746 (2.5%)


count    29254.000000
mean         5.117215
std          3.498609
min          0.000000
25%          3.000000
50%          4.500000
75%          6.500000
max         30.000000
Name: exp_avg, dtype: float64

**Fix vs. original version:** the original parser discarded any row with only a single experience number found (e.g. "5 yrs" flat, rather than a range), forcing it to NaN even though it's a perfectly parseable value. That silently lost valid rows for no reason — fixed here to handle both cases explicitly.

## 7. Location Cleaning

**Goal:** Extract a single primary city from free-text location fields, which sometimes list multiple cities.

**Documented limitation:** this notebook does a minimal, structural cleanup (primary city extraction + the two most common naming variants). Full city consolidation for modeling purposes (grouping Delhi/Noida/Gurgaon into one metro region, keeping only top-N cities, etc.) is a modeling decision made in Notebook 3, not here — kept separate so this notebook's output stays a faithful, minimally-opinionated cleaned dataset.

In [11]:
def clean_location(loc):
    if pd.isna(loc):
        return np.nan

    loc = loc.title().strip()
    cities = [c.strip() for c in loc.split(",")]

    # Most common raw naming inconsistencies observed in this dataset
    mapping = {
        "Bangalore": "Bengaluru",
        "Delhi Ncr": "Delhi",
    }

    primary_city = cities[0]
    primary_city = mapping.get(primary_city, primary_city)

    return primary_city


df["location_cleaned"] = df["location"].apply(clean_location)
df["location_cleaned"].value_counts().head(10)


location_cleaned
Bengaluru    6039
Mumbai       4339
Delhi        3127
Pune         2687
Hyderabad    2604
Gurgaon      2140
Chennai      1947
Kolkata      1456
Noida        1333
Ahmedabad    1234
Name: count, dtype: int64

## 8. Skill Text Cleaning

**Goal:** Normalize the free-text `key_skills` field (delimiter variation, casing, duplicates) into a clean, comma-separated skill list for later TF-IDF and matching use.

**Bug fix vs. original version:** the original regex `[^a-z0-9, ]` stripped every character outside plain alphanumerics — which silently destroyed real, meaningful skill tokens like `c++`, `c#`, and `.net`, since `+`, `#`, and `.` were removed. In a technical-jobs dataset, that corrupts exactly the skills most likely to matter. Fixed below to preserve them.

In [12]:
def clean_skills(skills):
    if pd.isna(skills):
        return np.nan

    skills = skills.lower()
    skills = skills.replace("|", ",")
    skills = skills.replace("/", ",")

    # Preserve characters meaningful inside real skill tokens: c++, c#, .net, node.js
    skills = re.sub(r"[^a-z0-9+#. ,]", "", skills)

    skill_list = [s.strip() for s in skills.split(",") if s.strip() != ""]
    skill_list = list(dict.fromkeys(skill_list))  # de-dup, preserve order

    return ", ".join(skill_list)


df["key_skills_cleaned"] = df["key_skills"].apply(clean_skills)

# Verify the fix: these tokens should now survive cleaning
sample_check = df[df["key_skills"].str.contains(r"C\+\+|C#|\.NET", case=False, na=False)]
print(f"Rows containing C++/C#/.NET in raw skills: {len(sample_check)}")
if len(sample_check) > 0:
    print("Sample cleaned output:")
    print(sample_check[["key_skills", "key_skills_cleaned"]].head(3).to_string())


Rows containing C++/C#/.NET in raw skills: 1624
Sample cleaned output:
                                                                                                     key_skills                                                                                          key_skills_cleaned
19                                                         C#| MS Dynamics CRM| Microsoft Dynamics| ms dynamics                                                        c#, ms dynamics crm, microsoft dynamics, ms dynamics
28                                         C#| Java| OOP| C++| Multithreading| UML| OOAD| OOPS| .Net| debugging                                        c#, java, oop, c++, multithreading, uml, ooad, oops, .net, debugging
63   Graphics| C++| Windows OS| Linux| high performance computing| Test cases| System software| Gaming| FreeBSD  graphics, c++, windows os, linux, high performance computing, test cases, system software, gaming, freebsd


## 9. Unique Value Counts & Summary Statistics

In [13]:
for col in df.columns:
    print(f"{col:30s}: {df[col].nunique()} unique values")


uniq_id                       : 30000 unique values
crawl_timestamp               : 28865 unique values
job_title                     : 23884 unique values
job_salary                    : 1167 unique values
job_experience_required       : 256 unique values
key_skills                    : 26909 unique values
role_category                 : 206 unique values
location                      : 2573 unique values
functional_area               : 72 unique values
industry                      : 122 unique values
role                          : 649 unique values
salary_disclosed              : 2 unique values
salary_min                    : 72 unique values
salary_max                    : 99 unique values
salary_avg                    : 185 unique values
exp_min                       : 25 unique values
exp_max                       : 33 unique values
exp_avg                       : 54 unique values
location_cleaned              : 653 unique values
key_skills_cleaned            : 26726 unique val

In [14]:
df.describe()

,salary_disclosed,salary_min,salary_max,salary_avg,exp_min,exp_max,exp_avg
count,30000.000000,8.032000e+03,8.032000e+03,8.032000e+03,29254.000000,29254.000000,29254.000000
mean,0.307967,4.480108e+05,7.668154e+05,6.074131e+05,3.354618,6.879811,5.117215
std,0.461660,5.501575e+05,7.969626e+05,6.642255e+05,3.131717,3.995628,3.498609
min,0.000000,1.000000e+04,1.000000e+04,1.000000e+04,0.000000,0.000000,0.000000
25%,0.000000,1.750000e+05,3.000000e+05,2.500000e+05,1.000000,5.000000,3.000000
50%,0.000000,2.500000e+05,5.000000e+05,3.750000e+05,2.000000,6.000000,4.500000
75%,1.000000,5.000000e+05,9.000000e+05,7.000000e+05,5.000000,9.000000,6.500000
max,1.000000,1.000000e+07,1.000000e+07,1.000000e+07,30.000000,35.000000,30.000000


## 10. Sanity-Check Assertions

Before saving, verify the cleaned data doesn't violate basic logical constraints. These assertions are the difference between "the code ran without errors" and "the output is actually trustworthy" — if any of these fail, the notebook stops here rather than silently saving bad data downstream.

In [15]:
errors = []

# uniq_id should be unique
if df["uniq_id"].duplicated().any():
    errors.append(f"Duplicate uniq_id values found: {df['uniq_id'].duplicated().sum()}")

# salary_min should never exceed salary_max
bad_salary = df[df["salary_min"] > df["salary_max"]]
if len(bad_salary) > 0:
    errors.append(f"{len(bad_salary)} rows have salary_min > salary_max")

# salary values should be non-negative
if (df[["salary_min", "salary_max", "salary_avg"]].dropna() < 0).any().any():
    errors.append("Negative salary values found")

# salary magnitude should be plausible for an annual figure — catches parsing
# failures that are internally consistent (min <= max) but still garbage
# (e.g. a phone number parsed as a salary range). This check is deliberately
# separate from the internal-consistency checks above: a value can pass
# min <= max and still not plausibly be a real salary.
implausible_salary = df[df["salary_avg"] > 10_000_000]
if len(implausible_salary) > 0:
    errors.append(f"{len(implausible_salary)} rows have an implausible salary_avg (> Rs. 1 crore/year)")

# exp_min should never exceed exp_max
bad_exp = df[df["exp_min"] > df["exp_max"]]
if len(bad_exp) > 0:
    errors.append(f"{len(bad_exp)} rows have exp_min > exp_max")

# salary_disclosed flag should be internally consistent with parsed salary
inconsistent = df[(df["salary_disclosed"] == 0) & (df["salary_min"].notna())]
if len(inconsistent) > 0:
    errors.append(f"{len(inconsistent)} rows flagged as undisclosed but have a parsed salary value")

if errors:
    print("SANITY CHECKS FAILED:")
    for e in errors:
        print(" -", e)
else:
    print("All sanity checks passed.")


All sanity checks passed.


## 11. Save Cleaned Dataset

In [16]:
processed_dir = PROJECT_ROOT / "dataset"
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "naukri_cleaned.csv"
df.to_csv(output_path, index=False)

print(f"Cleaned dataset saved to: {output_path}")
print(f"Final shape: {df.shape}")


Cleaned dataset saved to: c:\Users\venut\OneDrive\Desktop\job_AI_system\dataset\naukri_cleaned.csv
Final shape: (30000, 20)


## 12. Summary

**What this notebook did:**
- Loaded 30000 raw job postings and standardized column names
- Parsed free-text salary into structured numeric fields; found that only a subset of postings have a usable numeric salary (see Section 5 output for the exact rate — quoting it here as a fixed number would go stale if the parser changes)
- Caught and fixed two real data corruption cases via the Section 10 sanity checks: (1) stray concatenated digits producing `salary_min > salary_max`, and (2) phone numbers and unrelated free text being parsed as salaries in the tens/hundreds of millions — the second was only discovered because it silently destroyed the experience-salary correlation signal in Notebook 2, which is the actual reason a magnitude-plausibility check exists here now, not just an internal-consistency check
- Parsed free-text experience requirements into structured numeric fields, fixing a bug where single-value entries were previously discarded
- Cleaned location text to a primary-city field
- Fixed a skill-cleaning bug that was destroying meaningful technical tokens (C++, C#, .NET)
- Ran explicit sanity-check assertions on the cleaned output before saving

**Key limitations carried forward to later notebooks:**
- Salary is only usable for the minority of postings that disclosed it — any model trained on salary is therefore trained on a non-random subset of the data (selection bias). This must be restated wherever the salary model is used, not just here.
- Annual vs. monthly salary convention is assumed, not verified per posting.
- `industry` field contamination (long company-description text mixed into what should be a short category) is identified but **not fixed here** — it's addressed in Notebook 2 (where it's diagnosed) and Notebook 3 (where it's fixed), consistent with keeping this notebook focused on structural parsing rather than modeling-oriented category cleanup.

**Next notebook:** `2_exploratory_data_analysis.ipynb` — analyze the distributions and relationships in this cleaned dataset.
